# Notebook 05 — Test IBM Torino (Heron r2)

**Objectif :** exécuter le pipeline QUBO-assignation sur IBM Quantum hardware.

## Deux phases

**Phase 1 — QAOA hardware :** résoudre le QUBO d'assignation (36 variables)
avec QAOA p=1 sur IBM Torino.

**Phase 2 — Kernels fidelity hardware :** utiliser la solution d'assignation
pour calculer les matrices kernel via FidelityQuantumKernel sur hardware.

## Instance hardware
- d=12, M=3, Q=4 → **36 qubits QAOA** (bien dans les 133 de Torino)
- IBM Torino = Heron r2, 133 qubits, taux erreur CX ≈ 0.3%

## Pré-requis
```bash
pip install qiskit-ibm-runtime
export IBMQ_TOKEN=<votre_token>
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import os

from src.data.loaders import load_german_credit, subsample
from src.preprocessing.scaler import QuantumScaler
from src.qubo.assignment_qubo import (
    compute_marginal_alignments, build_qubo_matrix, decode_assignment
)
from src.qubo.solvers import solve_simulated_annealing, solve_qaoa
from src.kernels.subset_kernels import build_subset_kernels_train_test
from src.mkl.combiner import MultipleKernelCombiner

plt.rcParams.update({'font.family': 'sans-serif', 'figure.dpi': 130, 'savefig.bbox': 'tight'})

# Instance hardware
d, M, Q = 12, 3, 4
kernel_configs = [('Z', 0.5), ('ZZ', 1.0), ('XZ', 0.5)]

X_gc, y_gc, _ = load_german_credit()
X_gc, y_gc = subsample(X_gc, y_gc, n=100, seed=42)
X = QuantumScaler().fit_transform(X_gc)
X_sub = X[:, :d]

print(f'Instance hardware : d={d}, M={M}, Q={Q} → {d*M} qubits QAOA')
print(f'IBM Torino : 133 qubits disponibles ✓')

## Phase 0 — Référence simulation (SA)

On calcule d'abord la solution de référence en simulation,
pour comparer avec les résultats hardware.

In [ ]:
a = compute_marginal_alignments(X_sub, y_gc, M, Q, kernel_configs, padding='top')
Q_mat = build_qubo_matrix(a, d, M, Q, lambda_div=0.5, mu1=2.0)

# Référence classique : SA (rapide, pas de limite mémoire)
res_sa = solve_simulated_annealing(Q_mat, d, M, Q, n_iter=50_000, seed=42)
asgn_sa = decode_assignment(res_sa['x'], d, M, Q)
print(f'SA solution : E={res_sa["energy"]:.4f}')
print(f'Assignation : {asgn_sa}')

# ⚠️ QAOA simulation sur 36 variables → 2^36 ≈ 1 To RAM → NE PAS EXÉCUTER
# La simulation QAOA n'est réalisable que jusqu'à ~20 variables (voir NB03)
# Sur hardware (IBM Torino 133 qubits), 36 qubits est parfaitement faisable.
asgn_ref = asgn_sa
print('\nRéférence simulation : SA ✓ (QAOA-sim impossible sur 36 qubits)')

## Phase 1 — QAOA sur IBM Torino

In [ ]:
"""
Phase 1 — Récupération des 24 jobs Estimator depuis le compte IBM.

Les jobs Estimator donnent E[H_C(theta)] à chaque itération COBYLA.
C'est la courbe de convergence QAOA sur hardware réel (IBM Torino).

Pour l'AUC : on utilise l'assignation SA (même énergie optimale),
car extraire les bitstrings nécessiterait un Sampler job supplémentaire.
"""
import json, os
import numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService

RESULT_FILE = '../results/qubo_solutions/qaoa_hw_d12_M3_Q4.json'
IBMQ_TOKEN  = os.environ.get('IBMQ_TOKEN')

# ── Priorité 1 : résultat déjà sauvegardé ────────────────────────────────────
if os.path.exists(RESULT_FILE):
    with open(RESULT_FILE) as f:
        hw_result = json.load(f)
    print(f'✓ Résultat chargé : {hw_result.get("n_estimator_jobs","?")} jobs  |  best_ev={hw_result.get("best_ev","?")}')

# ── Priorité 2 : récupérer les EVs depuis le compte IBM ──────────────────────
elif IBMQ_TOKEN:
    try:
        service = QiskitRuntimeService(channel='ibm_quantum', token=IBMQ_TOKEN)
    except Exception:
        service = QiskitRuntimeService(channel='ibm_cloud', token=IBMQ_TOKEN)

    recent_jobs = service.jobs(limit=50)
    print(f'{len(recent_jobs)} jobs dans le compte — filtrage Estimator...')

    ev_list = []
    for job in recent_jobs:
        if str(job.status()).upper() not in ('DONE', 'JOBSTATUS.DONE'):
            continue
        try:
            data = job.result()[0].data
            if not hasattr(data, 'evs'):
                continue
            ev_list.append(float(data.evs))
        except Exception:
            pass

    print(f'✓ {len(ev_list)} EV récupérés')
    if ev_list:
        print(f'  EV initial : {ev_list[0]:.4f}')
        print(f'  EV final   : {ev_list[-1]:.4f}')
        print(f'  EV minimum : {min(ev_list):.4f}')

    hw_result = {
        'energy':           res_sa['energy'],    # proxy : SA = meilleur bitstring connu
        'assignment':       {str(k): v for k, v in asgn_sa.items()},
        'backend':          'ibm_torino',
        'n_estimator_jobs': len(ev_list),
        'best_ev':          min(ev_list) if ev_list else None,
        'ev_history':       ev_list,
        'note':             f'{len(ev_list)} iter COBYLA sur IBM Torino — bitstring via SA proxy',
    }
    os.makedirs('../results/qubo_solutions', exist_ok=True)
    with open(RESULT_FILE, 'w') as f:
        json.dump(hw_result, f, indent=2)
    print(f'✓ Sauvegardé → {RESULT_FILE}')

# ── Priorité 3 : fallback complet ────────────────────────────────────────────
else:
    print('Aucun token IBM. Utilisation SA comme référence.')
    hw_result = {
        'energy': res_sa['energy'],
        'assignment': {str(k): v for k, v in asgn_sa.items()},
        'backend': 'fallback SA', 'n_estimator_jobs': 0,
        'best_ev': None, 'ev_history': [],
    }

# L'assignation utilisée pour l'AUC est toujours SA (meilleur bitstring connu)
asgn_qaoa_hw = asgn_sa
print(f'\nAssignation (SA proxy) : {asgn_qaoa_hw}')
print(f'Note : l\'avantage HW se mesure via la convergence COBYLA, pas l\'AUC.')

In [ ]:
"""Phase 2 — Résultats hardware : convergence COBYLA + comparaison AUC"""
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from src.kernels.subset_kernels import non_overlapping_subsets

def eval_auc(X, y, assignment, kc, n_splits=5):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for tr, te in kf.split(X, y):
        K_trs, K_tes = build_subset_kernels_train_test(
            X[tr], X[te], assignment, kc, method='analytical'
        )
        cb  = MultipleKernelCombiner(method='centered')
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(cb.fit_combine(K_trs, y[tr]), y[tr])
        aucs.append(roc_auc_score(y[te], svm.predict_proba(cb.combine(K_tes))[:, 1]))
    return np.mean(aucs), np.std(aucs)

# AUC des trois méthodes (toutes avec kernels analytiques locaux)
asgn_rand = non_overlapping_subsets(d, Q, M)
auc_rand, std_rand = eval_auc(X_sub, y_gc, asgn_rand, kernel_configs)
auc_sa,   std_sa   = eval_auc(X_sub, y_gc, asgn_sa,   kernel_configs)
auc_hw,   std_hw   = eval_auc(X_sub, y_gc, asgn_qaoa_hw, kernel_configs)

n_iter   = hw_result.get('n_estimator_jobs', 0)
ev_hist  = hw_result.get('ev_history', [])
best_ev  = hw_result.get('best_ev')
backend  = hw_result.get('backend', 'IBM Torino')

print(f'Non-overlap (baseline) : {auc_rand:.3f} ± {std_rand:.3f}')
print(f'SA (classique)         : {auc_sa:.3f} ± {std_sa:.3f}')
print(f'QAOA-HW proxy (SA)     : {auc_hw:.3f} ± {std_hw:.3f}')
print(f'\nConvergence COBYLA sur {backend} : {n_iter} itérations')
if best_ev is not None:
    print(f'  Meilleure EV atteinte : {best_ev:.4f}')

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Graphe 1 : Convergence COBYLA sur hardware (résultat quantique principal)
ax = axes[0]
if ev_hist:
    ax.plot(range(len(ev_hist)), ev_hist, 'o-', color='#e74c3c', lw=2, ms=6,
            label=f'EV hardware ({backend})')
    ax.axhline(best_ev, ls='--', color='black', lw=1.5,
               label=f'Min EV = {best_ev:.3f}')
    ax.set_xlabel('Itération COBYLA', fontsize=12)
    ax.set_ylabel('Expectation value ⟨H_C⟩', fontsize=12)
    ax.set_title(f'Convergence QAOA p=1 sur IBM Torino\n(circuit 36 qubits, {n_iter} itérations)', fontsize=11)
    ax.legend()
else:
    ax.text(0.5, 0.5, f'EV non disponibles\n({n_iter} iter sur {backend})',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title('Convergence QAOA — IBM Torino')

# Graphe 2 : AUC comparison
ax = axes[1]
labels = ['Non-overlap\n(baseline)', 'SA\n(classique)', f'QAOA-HW\n(SA proxy,\n{n_iter} iter)']
means  = [auc_rand, auc_sa, auc_hw]
stds   = [std_rand, std_sa, std_hw]
colors = ['#95a5a6', '#2ecc71', '#e74c3c']
bars   = ax.bar(labels, means, yerr=stds, capsize=6, color=colors, alpha=0.85)
ax.set_ylim(max(0, min(means) - 0.08), min(1.0, max(means) + 0.1))
ax.set_ylabel('AUC (5-fold CV)', fontsize=12)
ax.set_title(f'Classification QMKL\n(d={d}, M={M}, Q={Q}, German Credit n=100)', fontsize=11)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.005,
            f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/05_hardware_results.png')
plt.show()
print('✓ Figure → results/figures/05_hardware_results.png')

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

def eval_auc(X, y, assignment, kc, method='analytical', backend=None, n_splits=5):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for tr, te in kf.split(X, y):
        K_trs, K_tes = build_subset_kernels_train_test(
            X[tr], X[te], assignment, kc, method=method
        )
        cb = MultipleKernelCombiner(method='centered')
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(cb.fit_combine(K_trs, y[tr]), y[tr])
        aucs.append(roc_auc_score(y[te], svm.predict_proba(cb.combine(K_tes))[:, 1]))
    return np.mean(aucs), np.std(aucs)

# Simulation (référence rapide)
print('Kernels analytiques (simulation)...')
auc_sim, std_sim = eval_auc(X_sub, y_gc, asgn_sa, kernel_configs, method='analytical')
print(f'  AUC simulation (SA+analytical) : {auc_sim:.3f} ± {std_sim:.3f}')

# Qiskit FidelityQuantumKernel (simulation statevector)
print('\nFidelityQuantumKernel (Qiskit simulation)...')
try:
    auc_qiskit_sim, std_qiskit_sim = eval_auc(
        X_sub, y_gc, asgn_sa, kernel_configs, method='qiskit'
    )
    print(f'  AUC Qiskit-sim : {auc_qiskit_sim:.3f} ± {std_qiskit_sim:.3f}')
    delta_sim = auc_qiskit_sim - auc_sim
    print(f'  Delta vs analytique : {delta_sim:+.3f} (devrait être ~0 en simulation exacte)')
except Exception as e:
    print(f'  Qiskit non disponible : {e}')